# 직접 수집 3 · 데이터 수집 ② — XTools 지표

필터링된 각 문서의 **활동 지표**를 수집합니다. 지표 원천은 **XTools**(위키미디어 통계 도구) + 위키 API:
- **편집수(edit)** · **조회수(pageviews)** · **링크(link)** · **문서 정보(info: 생성일·분류 등)**

이 값들이 나중에 **기술집약도·수요/공급 부상성** 지표로 환산됩니다.
> ⚠️ 문서마다 4종 호출이 있어 **가장 오래 걸리는 단계**입니다(수 분). 이미 수집된 경우 캐시로 건너뜁니다.

In [ ]:
# --- 공통 준비: 프로젝트 루트로 이동 + 임포트 + 경로 정의 ---
import os, sys, warnings
warnings.filterwarnings("ignore")
for _ in range(5):                       # pipeline.py 있는 프로젝트 루트 탐색
    if os.path.isfile("pipeline.py"):
        break
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import pandas as pd
import pipeline as P
import wiki_crawling as wc

SEED = "Quantum computing"               # ← 분석할 위키 문서 (자유 변경)
N_DEPTH = 1                              # 1차수 고정
slug = P.slugify_seed(SEED)
BASE = os.path.join("runs", slug)
os.makedirs(os.path.join(BASE, "seed_item"), exist_ok=True)
os.makedirs(os.path.join(BASE, "xtools_item", slug), exist_ok=True)
EXPAND   = os.path.join(BASE, "seed_item", f"{N_DEPTH}차시 확장 최종_결과.xlsx")
FILTER   = os.path.join(BASE, "seed_item", f"{slug}_filtering_network.xlsx")
XTOOLS   = os.path.join(BASE, "xtools_item", slug)
FLAG     = os.path.join(XTOOLS, ".collect_done")
PAGERANK = os.path.join(BASE, f"{slug}_pagerank.xlsx")
STATS    = os.path.join(BASE, f"{slug}_statistics.xlsx")
print("작업 루트:", os.getcwd())
print("대상 문서:", SEED, "| 작업 폴더:", BASE)

In [ ]:
# [이전 단계 재현] 시드 실제 제목 (캐시되어 즉시)
df_true = P.resolve_true_title(SEED, os.path.join(BASE, "true_title.xlsx"))
SEED_TRUE = P.title_space(str(df_true.loc[0, "True_title"]))
print("시드 실제 제목:", SEED_TRUE)

In [ ]:
# [이전 단계 재현] 네트워크 확장 (캐시되어 즉시)
EXPAND = P.expand_network(SEED_TRUE, EXPAND, N_DEPTH)
print("확장 결과:", EXPAND)

In [ ]:
# [이전 단계 재현] 네트워크 필터링 (캐시되어 즉시)
FILTER = P.filter_network(EXPAND, SEED_TRUE, FILTER, mode="balanced")
print("필터 결과:", FILTER)

## 1) 개별 XTools 호출 맛보기 — 시드의 연도별 편집수
전체 수집 전에, 한 문서의 편집 이력을 직접 불러 API 형태를 봅니다.

In [ ]:
edit_df = wc.crawl_edit_by_year(SEED_TRUE)
print(f"'{SEED_TRUE}' 연도별 편집 데이터")
edit_df.head(12)

## 2) 전체 노드 수집 + 통합
`xtools_collect` 가 필터된 모든 노드의 4종 지표를 수집하고(문서별 파일),
`xtools_integrate` 가 이를 지표별 통합본으로 합칩니다.

In [ ]:
def on_node(**kw):
    print(f"  수집: {kw.get('node')}")

P.xtools_collect(SEED_TRUE, FILTER, XTOOLS, FLAG, on_node=on_node)
outs = P.xtools_integrate(SEED_TRUE, XTOOLS)
print("\n통합 파일:", {k: os.path.basename(v) for k, v in outs.items()})

## 3) 통합 결과 살펴보기 (편집 통합본 예시)

In [ ]:
edit_all = pd.read_excel(outs["edit"]) if "edit" in outs else pd.DataFrame()
print("편집 통합본 shape:", edit_all.shape)
edit_all.head(10)

**정리**
- 연관 문서 전체의 편집·조회·링크·정보를 수집·통합했습니다.
- 다음 노트북(**04 결과**)에서 이 원천 데이터를 지표로 환산하고 네트워크를 분석합니다.